# Phase 1.7: Recording-split v2a-RSN Bootstrap Null Calibration — `220119_F2_run11`

This notebook calibrates the observed graph-instability statistic for one v2a-RSN recording across the primary c-GC methods.

Workflow:

1. Load the completed `220119_F2_run11.pkl` adjacency grid for each method.
2. Load the exact filtered raw traces recorded by the shared v2a profile.
3. Re-estimate each method on moving-block bootstrap surrogate traces using the parameters from `run_metadata.json`.
4. Save every surrogate adjacency grid as an independent checkpoint, allowing interrupted runs to resume.
5. Export recording-specific critical values, pointwise depth bands, summary tables, and figures.

All artifacts are isolated below `outputs/calibration/v2a/{analysis_profile}/220119_F2_run11/`. This means the four recording notebooks can run concurrently without sharing mutable output files. Within this notebook, methods are run sequentially so each method keeps its own checkpoint directory. The depth grid remains `p=1,...,7`.

The notebook does not read `transitions.csv`; depth values come from the completed pickle dictionaries.


In [ ]:
from __future__ import annotations

import json
import logging
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError(
            'Could not find the hidden-confounding-diagnostics project root'
        )
    PROJECT_ROOT = parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.v2a_calibration import (  # noqa: E402
    calibration_payload,
    load_v2a_calibration_input,
    make_v2a_surrogate_analyzer,
    plot_v2a_calibration_grid,
    plot_v2a_pointwise_grid,
    run_resumable_v2a_calibration,
    stable_seed,
    write_calibration_payload,
)
from markovianity_diagnostic.experiments.v2a_rsn_utils import (  # noqa: E402
    V2A_ANALYSIS_PROFILE,
)

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
ANALYSIS_PROFILE = V2A_ANALYSIS_PROFILE
RECORDING = '220119_F2_run11'
METHOD_SPECS = [
    {'method_dir': 'c-GC', 'method_label': 'c-GC'},
    {'method_dir': 'c-GC-star', 'method_label': 'c-GC*'},
]
OUTPUT_DIR = (
    PROJECT_ROOT
    / 'outputs'
    / 'calibration'
    / 'v2a'
    / ANALYSIS_PROFILE
    / RECORDING
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

P_VALUES = [1, 2, 3, 4, 5, 6, 7]
P0 = 1
B = 1
BLOCK_LENGTH = 50
SEED = 42
N_JOBS = 1
SHOW_PROGRESS = True

# B is a cumulative target. Rerunning with a larger B reuses compatible
# replicate checkpoints and computes only the missing replicate indices.
print(f'Output directory: {OUTPUT_DIR}')
print(f'Recording: {RECORDING}')
print('Methods: ' + ', '.join(spec['method_label'] for spec in METHOD_SPECS))
print(f'Depths: {P_VALUES}; p0={P0}')
print(f'Bootstrap: B={B}, block_length={BLOCK_LENGTH}, n_jobs={N_JOBS}')


In [ ]:
method_inputs = {}
for spec in METHOD_SPECS:
    method_dir = spec['method_dir']
    print(f'Validating {method_dir}/{RECORDING} inputs...')
    calibration_input = load_v2a_calibration_input(
        PROJECT_ROOT,
        recording=RECORDING,
        method_dir=method_dir,
        p_values=P_VALUES,
        analysis_profile=ANALYSIS_PROFILE,
    )
    method_inputs[method_dir] = calibration_input
    print(
        f"  connectivity={calibration_input.connectivity_path.relative_to(PROJECT_ROOT)}; "
        f"X={calibration_input.X.shape}"
    )

print(f'Validated {len(method_inputs)} method input sets for {RECORDING}.')


In [ ]:
results = {spec['method_dir']: {} for spec in METHOD_SPECS}
summary_rows = []
run_output_paths = []
input_paths = []
suite_start = time.perf_counter()

for method_index, spec in enumerate(METHOD_SPECS, start=1):
    method_dir = spec['method_dir']
    method_label = spec['method_label']
    print(
        f'[{method_index}/{len(METHOD_SPECS)}] {RECORDING}: {method_label}'
    )
    run_start = time.perf_counter()

    calibration_input = method_inputs[method_dir]
    analyze_surrogate = make_v2a_surrogate_analyzer(calibration_input.metadata)
    run_seed = stable_seed(SEED, RECORDING, method_dir)
    method_output_dir = OUTPUT_DIR / method_dir
    checkpoint_dir = method_output_dir / 'checkpoints'
    learner_params = (
        calibration_input.metadata.get('gcstar_params')
        or calibration_input.metadata.get('tigramite_params')
    )
    if learner_params is None:
        raise ValueError(
            f'{method_dir}/{RECORDING} metadata has no supported learner parameters'
        )
    config_metadata = {
        'recording': RECORDING,
        'method_dir': method_dir,
        'analysis_profile': ANALYSIS_PROFILE,
        'trace_selection': calibration_input.metadata['trace_selection'],
        'learner_params': learner_params,
        'inferred_completion': calibration_input.metadata.get('inferred_completion'),
    }

    outcome = run_resumable_v2a_calibration(
        calibration_input.X,
        observed_adjacencies=calibration_input.observed_adjacencies,
        analyze_surrogate=analyze_surrogate,
        p_values=P_VALUES,
        p0=P0,
        B=B,
        block_length=BLOCK_LENGTH,
        seed=run_seed,
        checkpoint_dir=checkpoint_dir,
        n_jobs=N_JOBS,
        config_metadata=config_metadata,
        show_progress=SHOW_PROGRESS,
        progress_label=f'{RECORDING} {method_label}',
        progress_position=method_index - 1,
    )

    elapsed = time.perf_counter() - run_start
    run_metadata = {
        **config_metadata,
        'connectivity_pickle': str(
            calibration_input.connectivity_path.relative_to(PROJECT_ROOT)
        ),
        'trace_shape_time_by_neurons': list(calibration_input.X.shape),
        'p_values': P_VALUES,
        'p0': P0,
        'B': B,
        'block_length': BLOCK_LENGTH,
        'seed': run_seed,
        'elapsed_seconds': elapsed,
    }
    payload = calibration_payload(outcome, metadata=run_metadata)
    run_output_path = method_output_dir / 'bootstrap.json'
    write_calibration_payload(run_output_path, outcome, metadata=run_metadata)
    run_output_paths.append(run_output_path)
    results[method_dir][RECORDING] = payload
    input_paths.append(
        str(calibration_input.connectivity_path.relative_to(PROJECT_ROOT))
    )

    summary_rows.append({
        'recording': RECORDING,
        'method': method_dir,
        'method_label': method_label,
        'T_obs': outcome.result.observed['T_obs'],
        'critical_90': outcome.result.null['critical_90'],
        'critical_95': outcome.result.null['critical_95'],
        'critical_99': outcome.result.null['critical_99'],
        'p_value': outcome.result.null['p_value'],
        'reject_global_95': outcome.result.diagnosis['reject_global_95'],
        'first_exceedance_depth': outcome.result.diagnosis['first_exceedance_depth'],
        'B': B,
        'reused_replicates': outcome.reused_replicates,
        'elapsed_seconds': elapsed,
    })
    print(
        f"  T_obs={outcome.result.observed['T_obs']:.6f}; "
        f"critical_95={outcome.result.null['critical_95']:.6f}; "
        f"p={outcome.result.null['p_value']:.4f}; "
        f"reused={outcome.reused_replicates}/{B}; elapsed={elapsed:.1f}s"
    )

for trace_path in sorted(
    (PROJECT_ROOT / 'data' / 'v2a-RSNs' / RECORDING).glob(
        '*cells_fluorescence_signals.npy'
    )
):
    input_paths.append(str(trace_path.relative_to(PROJECT_ROOT)))

suite_elapsed = time.perf_counter() - suite_start
print()
print(f'{RECORDING} calibration completed in {suite_elapsed:.1f}s')


In [ ]:
bootstrap_results_path = OUTPUT_DIR / 'bootstrap_results.json'
bootstrap_results_tmp = OUTPUT_DIR / '.bootstrap_results.json.tmp'
bootstrap_results_tmp.write_text(
    json.dumps(
        {
            'schema_version': 3,
            'analysis': 'recording-split pickle-first v2a moving-block bootstrap calibration',
            'recording': RECORDING,
            'methods': results,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
bootstrap_results_tmp.replace(bootstrap_results_path)

summary_df = pd.DataFrame(summary_rows).sort_values(['recording', 'method'])
summary_path = OUTPUT_DIR / 'bootstrap_summary.csv'
summary_df.to_csv(summary_path, index=False)

histogram_path = plot_v2a_calibration_grid(
    results,
    OUTPUT_DIR / 'bootstrap_T_obs.png',
)
depth_bands_path = plot_v2a_pointwise_grid(
    results,
    OUTPUT_DIR / 'depth_bands.png',
)

print(summary_df.to_string(index=False))
print(f'Wrote {bootstrap_results_path}')
print(f'Wrote {summary_path}')
print(f'Wrote {histogram_path}')
print(f'Wrote {depth_bands_path}')


In [ ]:
is_full_production_run = B >= 50
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'bootstrap_null_v2a_recording_split_pickle_first',
    'analysis_profile': ANALYSIS_PROFILE,
    'status': 'complete' if is_full_production_run else 'smoke_or_subset',
    'input_contract': 'observed adjacency pickles plus raw traces for null fitting',
    'observed_connectivity_recomputed': False,
    'recording': RECORDING,
    'methods': [spec['method_dir'] for spec in METHOD_SPECS],
    'method_params': {
        'B': B,
        'p_values': P_VALUES,
        'p0': P0,
        'null_model': 'moving-block residual bootstrap',
        'block_length': BLOCK_LENGTH,
        'n_jobs': N_JOBS,
    },
    'random_seed': SEED,
    'input_paths': input_paths,
    'output_paths': [
        str(bootstrap_results_path.relative_to(PROJECT_ROOT)),
        str(summary_path.relative_to(PROJECT_ROOT)),
        str(histogram_path.relative_to(PROJECT_ROOT)),
        str(depth_bands_path.relative_to(PROJECT_ROOT)),
        *(str(path.relative_to(PROJECT_ROOT)) for path in run_output_paths),
    ],
    'software_versions': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'elapsed_seconds': suite_elapsed,
}
manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_tmp = OUTPUT_DIR / '.manifest.json.tmp'
manifest_tmp.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest_tmp.replace(manifest_path)
print(f'Wrote {manifest_path}')


In [ ]:
expected_outputs = [
    bootstrap_results_path,
    summary_path,
    histogram_path,
    depth_bands_path,
    manifest_path,
    *run_output_paths,
]
missing_outputs = [path for path in expected_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing calibration outputs: {missing_outputs}')

expected_rows = len(METHOD_SPECS)
assert len(summary_df) == expected_rows, (
    f'Expected {expected_rows} method rows, found {len(summary_df)}'
)
assert summary_df['recording'].eq(RECORDING).all()
assert summary_df['method'].isin([spec['method_dir'] for spec in METHOD_SPECS]).all()
assert summary_df['B'].eq(B).all()
assert summary_df['p_value'].between(0, 1).all()

for spec in METHOD_SPECS:
    method_dir = spec['method_dir']
    checkpoint_dir = OUTPUT_DIR / method_dir / 'checkpoints'
    checkpoint_count = len(list(checkpoint_dir.glob('replicate_*.pkl')))
    assert checkpoint_count >= B, (
        f'{method_dir}/{RECORDING} has {checkpoint_count} checkpoints; '
        f'expected at least {B}'
    )

print(f'Verified {len(expected_outputs)} output files and {expected_rows} rows.')
